In [2]:

import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import BernoulliNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

In [3]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("kazanova/sentiment140")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'sentiment140' dataset.
Path to dataset files: /kaggle/input/sentiment140


In [4]:
df=pd.read_csv('/kaggle/input/sentiment140/training.1600000.processed.noemoticon.csv',encoding='latin-1',header=None)
df=df[[0,5]]
df.columns=['polarity','text']
print(df.head())

   polarity                                               text
0         0  @switchfoot http://twitpic.com/2y1zl - Awww, t...
1         0  is upset that he can't update his Facebook by ...
2         0  @Kenichan I dived many times for the ball. Man...
3         0    my whole body feels itchy and like its on fire 
4         0  @nationwideclass no, it's not behaving at all....


In [8]:
print(df['polarity'].value_counts())

polarity
0.0    800000
Name: count, dtype: int64


In [7]:
df = df[df.polarity != 2]

df['polarity'] = df['polarity'].map({0: 0, 4: 1})

print(df['polarity'].value_counts())

polarity
0.0    800000
Name: count, dtype: int64


In [9]:
def clean_text(text):
  return text.lower()


df['clean_text']=df['text'].apply(clean_text)

print(df[['text','clean_text']].head())

                                                text  \
0  @switchfoot http://twitpic.com/2y1zl - Awww, t...   
1  is upset that he can't update his Facebook by ...   
2  @Kenichan I dived many times for the ball. Man...   
3    my whole body feels itchy and like its on fire    
4  @nationwideclass no, it's not behaving at all....   

                                          clean_text  
0  @switchfoot http://twitpic.com/2y1zl - awww, t...  
1  is upset that he can't update his facebook by ...  
2  @kenichan i dived many times for the ball. man...  
3    my whole body feels itchy and like its on fire   
4  @nationwideclass no, it's not behaving at all....  


In [15]:
X_train,X_test,y_train,y_test=train_test_split(
    df['clean_text'],
    df['polarity'],
    test_size=0.2,
    random_state=42
)

print("Train size:",len(X_train))
print("Test size:",len(X_test))
print("y_train:",len(y_train))

Train size: 1280000
Test size: 320000
y_train: 1280000


In [12]:
vectorizer = TfidfVectorizer(max_features=5000, ngram_range=(1,2))

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

print("TF-IDF shape (train):", X_train_tfidf.shape)
print("TF-IDF shape (test):", X_test_tfidf.shape)

TF-IDF shape (train): (1280000, 5000)
TF-IDF shape (test): (320000, 5000)


In [20]:
import numpy as np
print(type(y_train))
print(len(y_train))
print(np.isnan(y_train).sum())

<class 'pandas.core.series.Series'>
1280000
639494


In [21]:
y_train = np.nan_to_num(y_train)

In [25]:
y_test = np.nan_to_num(y_test)

In [26]:
bnb = BernoulliNB()
bnb.fit(X_train_tfidf,y_train)

bnb_pred = bnb.predict(X_test_tfidf)

print("Bernoulli Naive Bayes Accuracy:", accuracy_score(y_test, bnb_pred))
print("\nBernoulliNB Classification Report:\n", classification_report(y_test, bnb_pred))

Bernoulli Naive Bayes Accuracy: 1.0

BernoulliNB Classification Report:
               precision    recall  f1-score   support

         0.0       1.00      1.00      1.00    320000

    accuracy                           1.00    320000
   macro avg       1.00      1.00      1.00    320000
weighted avg       1.00      1.00      1.00    320000

